# Superstore Sales Analysis — SQL Assignment

**Objective:** Use Subqueries, CTEs, and Window Functions to analyze sales data from the Superstore dataset.

I'm using SQLite here through Python since its easier to set up than MySQL. The queries are all standard SQL so they work the same way.

In [1]:
import pandas as pd
import sqlite3

In [2]:
# setting up the database connection
conn = sqlite3.connect('superstore.db')
cursor = conn.cursor()
print("Connected to database")

Connected to database


In [3]:
# helper function so i dont have to keep writing the same boilerplate
def run_query(query):
    return pd.read_sql_query(query, conn)

## Step 1: Setup Data

First loading the csv into a raw table, then splitting it into 3 normalized tables — customers, orders, products.

In [4]:
# loading the csv
df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')

# clean up column names - remove spaces
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')

# put everything in superstore_raw first
df.to_sql('superstore_raw', conn, if_exists='replace', index=False)

print(f"Loaded {len(df)} rows into superstore_raw")
run_query("SELECT * FROM superstore_raw LIMIT 3")

Loaded 9994 rows into superstore_raw


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.582
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


### Creating the 3 tables

In [5]:
# drop if they already exist (so i can rerun this cell)
cursor.execute("DROP TABLE IF EXISTS orders")
cursor.execute("DROP TABLE IF EXISTS customers")
cursor.execute("DROP TABLE IF EXISTS products")

# customers table - keeping only the fields that are truly per-customer
# (city/state/region change per order since customers order from different locations)
cursor.execute('''
    CREATE TABLE customers (
        Customer_ID TEXT PRIMARY KEY,
        Customer_Name TEXT,
        Segment TEXT
    )
''')

# products table
cursor.execute('''
    CREATE TABLE products (
        Product_ID TEXT PRIMARY KEY,
        Category TEXT,
        Sub_Category TEXT,
        Product_Name TEXT
    )
''')

# orders table - this has the per-transaction data
cursor.execute('''
    CREATE TABLE orders (
        Row_ID INTEGER PRIMARY KEY,
        Order_ID TEXT,
        Order_Date TEXT,
        Ship_Date TEXT,
        Ship_Mode TEXT,
        Customer_ID TEXT,
        Product_ID TEXT,
        Sales REAL,
        Quantity INTEGER,
        Discount REAL,
        Profit REAL,
        FOREIGN KEY (Customer_ID) REFERENCES customers(Customer_ID),
        FOREIGN KEY (Product_ID) REFERENCES products(Product_ID)
    )
''')

conn.commit()
print("Tables created successfully")

Tables created successfully


In [6]:
# inserting data using SELECT DISTINCT

cursor.execute('''
    INSERT INTO customers
    SELECT DISTINCT Customer_ID, Customer_Name, Segment
    FROM superstore_raw
    GROUP BY Customer_ID
''')

cursor.execute('''
    INSERT INTO products
    SELECT DISTINCT Product_ID, Category, Sub_Category, Product_Name
    FROM superstore_raw
    GROUP BY Product_ID
''')

cursor.execute('''
    INSERT INTO orders
    SELECT Row_ID, Order_ID, Order_Date, Ship_Date, Ship_Mode,
           Customer_ID, Product_ID, Sales, Quantity, Discount, Profit
    FROM superstore_raw
''')

conn.commit()

# verify counts
print("Customers inserted:", run_query("SELECT COUNT(*) FROM customers").iloc[0,0])
print("Products inserted:", run_query("SELECT COUNT(*) FROM products").iloc[0,0])
print("Orders inserted:", run_query("SELECT COUNT(*) FROM orders").iloc[0,0])

Customers inserted: 793
Products inserted: 1862
Orders inserted: 9994


793 customers, 1862 products, 9994 orders — looks right.

---

## Step 2: SQL Queries

### Q1: Orders where sales > average sales (Subquery)

In [7]:
# first check the average
avg_sales = run_query("SELECT ROUND(AVG(Sales), 2) FROM orders").iloc[0,0]
print(f"Average sales value: {avg_sales}")

# now the actual query with subquery
q1 = run_query('''
    SELECT * FROM orders
    WHERE Sales > (SELECT AVG(Sales) FROM orders)
''')

print(f"Number of orders above average: {len(q1)}")
q1.head()

Average sales value: 229.86
Number of orders above average: 2360


,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Product_ID,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.960,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.940,3,0.00,219.5820
2,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.578,5,0.45,-383.0310
3,6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-FU-10001487,261.540,2,0.00,41.8464
4,10,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,OFF-AP-10002311,1097.544,7,0.00,329.2632


2360 out of 9994 orders are above the average of ~$229.86. That makes sense because most orders are probably smaller items like office supplies and labels.

### Q2: Highest sales order for each customer (Subquery)

In [8]:
# using a correlated subquery here
q2 = run_query('''
    SELECT o.Row_ID, o.Order_ID, o.Customer_ID, o.Sales
    FROM orders o
    WHERE o.Sales = (
        SELECT MAX(o2.Sales)
        FROM orders o2
        WHERE o2.Customer_ID = o.Customer_ID
    )
''')

print(f"Total rows: {len(q2)}")
q2.head()

Total rows: 795


,Row_ID,Order_ID,Customer_ID,Sales
0,1,CA-2016-152156,CG-12520,731.94
1,4,US-2015-108966,SO-20335,957.5775
2,10,CA-2014-115812,BH-11710,1097.544
3,26,CA-2016-121755,KB-16585,4416.174
4,47,US-2015-118983,HP-14815,1044.63


795 rows instead of 793 customers — that's because a couple of customers have multiple orders tied at their max value. The correlated subquery checks each row against the max for that customer.

### Q3: Total sales for each customer (CTE)

In [9]:
q3 = run_query('''
    WITH CustomerTotalSales AS (
        SELECT Customer_ID, SUM(Sales) AS Total_Sales
        FROM orders
        GROUP BY Customer_ID
    )
    SELECT c.Customer_ID, c.Customer_Name, ROUND(cts.Total_Sales, 2) AS Total_Sales
    FROM CustomerTotalSales cts
    JOIN customers c ON c.Customer_ID = cts.Customer_ID
    ORDER BY Total_Sales DESC
''')

q3.head(10)

,Customer_ID,Customer_Name,Total_Sales
0,SM-20320,Sean Miller,25043.05
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.34
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57
5,KL-16645,Ken Lonsdale,14175.99
6,SC-20095,Sanjit Chand,14142.33
7,HL-15040,Hunter Lopez,12873.3
8,SE-20110,Sanjit Engle,12209.44
9,CC-12370,Christopher Conant,12129.07


Sean Miller is at the top with ~$25k. The CTE makes the query easier to read compared to doing it all in one nested query.

### Q4: Customers whose total sales are above average (CTE + Subquery)

In [10]:
# the CTE calculates totals, then the subquery in WHERE filters by average
q4 = run_query('''
    WITH CustomerTotalSales AS (
        SELECT Customer_ID, SUM(Sales) AS Total_Sales
        FROM orders
        GROUP BY Customer_ID
    )
    SELECT c.Customer_ID, c.Customer_Name, ROUND(cts.Total_Sales, 2) AS Total_Sales
    FROM CustomerTotalSales cts
    JOIN customers c ON c.Customer_ID = cts.Customer_ID
    WHERE cts.Total_Sales > (SELECT AVG(Total_Sales) FROM CustomerTotalSales)
    ORDER BY Total_Sales DESC
''')

# also get the average for reference
avg_total = run_query('''
    WITH CustomerTotalSales AS (
        SELECT Customer_ID, SUM(Sales) AS Total_Sales
        FROM orders
        GROUP BY Customer_ID
    )
    SELECT ROUND(AVG(Total_Sales), 2) FROM CustomerTotalSales
''').iloc[0,0]

print(f"Average total sales per customer: ${avg_total}")
print(f"Customers above average: {len(q4)} out of 793")
q4.head()

Average total sales per customer: $2297.23
Customers above average: 294 out of 793


,Customer_ID,Customer_Name,Total_Sales
0,SM-20320,Sean Miller,25043.05
1,TC-20980,Tamara Chand,19052.22
2,RB-19360,Raymond Buch,15117.34
3,TA-21385,Tom Ashbrook,14595.62
4,AB-10105,Adrian Barton,14473.57


294 out of 793 are above average which shows the sales distribution is pretty skewed — a few customers bring in a lot more revenue.

### Q5: Rank customers based on total sales (Window Function)

In [11]:
q5 = run_query('''
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        ROUND(SUM(o.Sales), 2) AS Total_Sales,
        RANK() OVER (ORDER BY SUM(o.Sales) DESC) AS Sales_Rank
    FROM orders o
    JOIN customers c ON c.Customer_ID = o.Customer_ID
    GROUP BY c.Customer_ID, c.Customer_Name
    ORDER BY Sales_Rank
''')

q5.head(10)

,Customer_ID,Customer_Name,Total_Sales,Sales_Rank
0,SM-20320,Sean Miller,25043.05,1
1,TC-20980,Tamara Chand,19052.22,2
2,RB-19360,Raymond Buch,15117.34,3
3,TA-21385,Tom Ashbrook,14595.62,4
4,AB-10105,Adrian Barton,14473.57,5
5,KL-16645,Ken Lonsdale,14175.99,6
6,SC-20095,Sanjit Chand,14142.33,7
7,HL-15040,Hunter Lopez,12873.3,8
8,SE-20110,Sanjit Engle,12209.44,9
9,CC-12370,Christopher Conant,12129.07,10


RANK() gives each customer a position. If two customers had the same total, they'd get the same rank and the next rank would be skipped.

### Q6: Row numbers for each order within a customer (Window Function + PARTITION BY)

In [12]:
q6 = run_query('''
    SELECT
        o.Customer_ID,
        c.Customer_Name,
        o.Order_ID,
        o.Sales,
        ROW_NUMBER() OVER (
            PARTITION BY o.Customer_ID
            ORDER BY o.Order_Date, o.Row_ID
        ) AS Order_Row_Num
    FROM orders o
    JOIN customers c ON c.Customer_ID = o.Customer_ID
    ORDER BY o.Customer_ID, Order_Row_Num
''')

q6.head(10)

,Customer_ID,Customer_Name,Order_ID,Sales,Order_Row_Num
0,AA-10315,Alex Avila,CA-2014-139451,3.400,1
1,AA-10315,Alex Avila,CA-2014-139451,3.920,2
2,AA-10315,Alex Avila,CA-2014-163459,95.616,3
3,AA-10315,Alex Avila,CA-2014-163459,113.328,4
4,AA-10315,Alex Avila,CA-2016-128084,15.552,5
5,AA-10315,Alex Avila,CA-2016-128084,407.976,6
6,AA-10315,Alex Avila,CA-2016-135545,19.440,7
7,AA-10315,Alex Avila,CA-2017-101663,4.980,8
8,AA-10315,Alex Avila,CA-2017-137330,25.824,9
9,AA-10315,Alex Avila,CA-2017-137330,85.056,10


PARTITION BY resets the row number counter for each customer. So Alex Avila's orders are numbered 1 through 10, and the next customer would start from 1 again. Pretty useful for seeing the order sequence per customer.

### Q7: Top 3 customers by total sales (Window Function)

In [13]:
# using DENSE_RANK so that if there are ties, we don't skip numbers
q7 = run_query('''
    WITH RankedCustomers AS (
        SELECT
            c.Customer_ID,
            c.Customer_Name,
            ROUND(SUM(o.Sales), 2) AS Total_Sales,
            DENSE_RANK() OVER (ORDER BY SUM(o.Sales) DESC) AS Sales_Rank
        FROM orders o
        JOIN customers c ON c.Customer_ID = o.Customer_ID
        GROUP BY c.Customer_ID, c.Customer_Name
    )
    SELECT * FROM RankedCustomers
    WHERE Sales_Rank <= 3
''')

q7

,Customer_ID,Customer_Name,Total_Sales,Sales_Rank
0,SM-20320,Sean Miller,25043.05,1
1,TC-20980,Tamara Chand,19052.22,2
2,RB-19360,Raymond Buch,15117.34,3


Sean Miller ($25k), Tamara Chand ($19k) and Raymond Buch ($15k) are the top 3.

---

## Step 3: Final Combined Query (JOIN + CTE + Window Function)

In [14]:
# combining everything: CTE for aggregation, JOIN for customer name, window function for rank
final_query = run_query('''
    WITH CustomerTotalSales AS (
        SELECT Customer_ID, SUM(Sales) AS Total_Sales
        FROM orders
        GROUP BY Customer_ID
    )
    SELECT
        c.Customer_Name,
        ROUND(cts.Total_Sales, 2) AS Total_Sales,
        RANK() OVER (ORDER BY cts.Total_Sales DESC) AS Sales_Rank
    FROM CustomerTotalSales cts
    JOIN customers c ON c.Customer_ID = cts.Customer_ID
    ORDER BY Sales_Rank
''')

final_query.head(15)

,Customer_Name,Total_Sales,Sales_Rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5
5,Ken Lonsdale,14175.99,6
6,Sanjit Chand,14142.33,7
7,Hunter Lopez,12873.3,8
8,Sanjit Engle,12209.44,9
9,Christopher Conant,12129.07,10


This final query shows everything together — CTE does the heavy lifting with aggregation, JOIN brings in the customer name, and RANK() assigns positions. Clean and readable.

---

## Mini Project: Customer Sales Insights

### 1. Who are the top 5 customers?

In [15]:
mp1 = run_query('''
    WITH RankedCustomers AS (
        SELECT
            c.Customer_Name,
            ROUND(SUM(o.Sales), 2) AS Total_Sales,
            RANK() OVER (ORDER BY SUM(o.Sales) DESC) AS Rank
        FROM orders o
        JOIN customers c ON c.Customer_ID = o.Customer_ID
        GROUP BY c.Customer_ID, c.Customer_Name
    )
    SELECT Customer_Name, Total_Sales, Rank
    FROM RankedCustomers
    WHERE Rank <= 5
''')
mp1

,Customer_Name,Total_Sales,Rank
0,Sean Miller,25043.05,1
1,Tamara Chand,19052.22,2
2,Raymond Buch,15117.34,3
3,Tom Ashbrook,14595.62,4
4,Adrian Barton,14473.57,5


### 2. Who are the bottom 5 customers?

In [16]:
# same thing but ranking ASC instead of DESC
mp2 = run_query('''
    WITH RankedCustomers AS (
        SELECT
            c.Customer_Name,
            ROUND(SUM(o.Sales), 2) AS Total_Sales,
            RANK() OVER (ORDER BY SUM(o.Sales) ASC) AS Rank
        FROM orders o
        JOIN customers c ON c.Customer_ID = o.Customer_ID
        GROUP BY c.Customer_ID, c.Customer_Name
    )
    SELECT Customer_Name, Total_Sales, Rank
    FROM RankedCustomers
    WHERE Rank <= 5
''')
mp2

,Customer_Name,Total_Sales,Rank
0,Jeffrey Brumfield,3.52,1
1,Li Peltz,8.56,2
2,Roy Collins,15.25,3
3,Grant Thornton,19.05,4
4,Don Reichenbach,21.78,5


Jeffrey Brumfield spent only $3.52 total. Thats a massive gap compared to Sean Miller's $25k at the top.

### 3. Which customers made only one order?

In [17]:
mp3 = run_query('''
    WITH OrderCounts AS (
        SELECT Customer_ID, COUNT(DISTINCT Order_ID) AS Num_Orders
        FROM orders
        GROUP BY Customer_ID
    )
    SELECT c.Customer_ID, c.Customer_Name, oc.Num_Orders
    FROM OrderCounts oc
    JOIN customers c ON c.Customer_ID = oc.Customer_ID
    WHERE oc.Num_Orders = 1
    ORDER BY c.Customer_Name
''')

print(f"Customers with only 1 order: {len(mp3)}")
mp3

Customers with only 1 order: 12


,Customer_ID,Customer_Name,Num_Orders
0,CC-12310,Chloris Kastensmidt,1
1,DR-12670,Don Reichenbach,1
2,GT-14635,Grant Thornton,1
3,JB-15745,Jeffrey Brumfield,1
4,JL-15835,Jim Loo,1
5,LP-17045,Li Peltz,1
6,MC-17395,MariaCard,1
7,NS-18235,Nat Simmerman,1
8,RC-19345,Roy Collins,1
9,SR-20485,Skye Rosenblum,1


12 customers only ever placed a single order. A few of these (Jeffrey Brumfield, Li Peltz, Roy Collins, Grant Thornton) are also in the bottom 5 by total sales, which makes sense — one small order = low total.

### 4. Which customers have above-average sales?

In [18]:
mp4 = run_query('''
    WITH CustomerTotalSales AS (
        SELECT Customer_ID, SUM(Sales) AS Total_Sales
        FROM orders
        GROUP BY Customer_ID
    )
    SELECT c.Customer_Name, ROUND(cts.Total_Sales, 2) AS Total_Sales
    FROM CustomerTotalSales cts
    JOIN customers c ON c.Customer_ID = cts.Customer_ID
    WHERE cts.Total_Sales > (SELECT AVG(Total_Sales) FROM CustomerTotalSales)
    ORDER BY cts.Total_Sales DESC
''')

print(f"{len(mp4)} customers have above-average total sales")
mp4.head(10)

294 customers have above-average total sales


,Customer_Name,Total_Sales
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57
5,Ken Lonsdale,14175.99
6,Sanjit Chand,14142.33
7,Hunter Lopez,12873.3
8,Sanjit Engle,12209.44
9,Christopher Conant,12129.07


### 5. Highest order value per customer

In [19]:
mp5 = run_query('''
    SELECT
        c.Customer_Name,
        MAX(o.Sales) AS Highest_Order_Value
    FROM orders o
    JOIN customers c ON c.Customer_ID = o.Customer_ID
    GROUP BY c.Customer_ID, c.Customer_Name
    ORDER BY Highest_Order_Value DESC
''')

mp5.head(10)

,Customer_Name,Highest_Order_Value
0,Sean Miller,17499.950
1,Tamara Chand,8399.976
2,Greg Tran,7999.980
3,Hunter Lopez,6999.960
4,Tom Ashbrook,6889.776
5,Ken Lonsdale,6879.960
6,Sanjit Chand,6719.952
7,Harry Greene,6567.858
8,Nathan Mautz,6205.600
9,Patrick O'Brill,5906.220


Sean Miller's biggest single order was $17.5k which is a huge chunk of his $25k total. So most of his total sales came from just one massive order. Interesting that Greg Tran shows up at #3 here but wasn't in the top 5 overall — he had one big order but not many others.

---

## Summary

**What I used and when:**
- **Subqueries** — filtering rows against an aggregated value (like avg or max). Works well for simple comparisons
- **CTEs** — breaking down complex queries into readable named blocks. Way easier to debug than deeply nested subqueries
- **Window Functions** — RANK(), ROW_NUMBER(), DENSE_RANK() for ordering and numbering without collapsing the result set the way GROUP BY does

**Key findings:**
- Sean Miller is the #1 customer at $25k, mostly driven by a single $17.5k order
- Jeffrey Brumfield is at the very bottom with just $3.52 total
- Only 294 out of 793 customers (about 37%) are above the average total sales of ~$2297
- 12 customers placed just a single order — most of these overlap with the lowest spenders
- The data is heavily right-skewed with a small number of high-value customers pulling the average up

In [20]:
conn.close()